# Notebook 01: LTM Insurance Data Loading & Scaling Instances

This notebook loads the **LTM formal specification** data (20 coverages across 9 families,
10 packages) and creates the scaling hierarchy of `BundlingProblem` instances used by the
DQI vs QAOA comparison pipeline.

The LTM data comes from the |Y>Quantum 2026 hackathon challenge (Travelers + Quantinuum + LTM)
and represents a realistic P&C (Property & Casualty) insurance product bundling scenario.

**Inputs**: `LTM/YQH26_data/` (6 CSV files from LTM consulting)
**Outputs**: `data/ltm_coverages.csv`, `data/ilp_n{n}.npz` for each scaling instance

In [ ]:
import sys
sys.path.insert(0, "..")

import json
import numpy as np
import pandas as pd
from pathlib import Path
from src.insurance_model import (
    InsuranceCoverage,
    BundlingProblem,
    CompatibilityRule,
    DependencyRule,
    solve_ilp,
    get_ilp_matrices,
    load_ltm_instance,
    subsample_problem,
)

## 1. Load LTM Insurance Data

The LTM formal specification defines **20 P&C insurance coverages** across **9 families**:
- **auto_base** (2 coverages, mandatory): basic vs enhanced liability
- **property_base** (3 coverages, mandatory): homeowners, condo, renters
- **auto_physical** (2 coverages): collision, comprehensive
- **auto_medical** (2 coverages): medical payments, personal injury protection
- **motorist_protect** (2 coverages): uninsured/underinsured motorist
- **auto_ancillary** (3 coverages): roadside, rental reimbursement, gap insurance
- **property_addon** (2 coverages): personal property floater, additional living expense
- **liability_extend** (2 coverages): personal umbrella, excess liability
- **specialty_peril** (2 coverages): flood, earthquake

Each coverage has a standalone price (annual premium), take rate, contribution margin, and
mandatory/optional designation within its family.

In [ ]:
# Load the full LTM instance from the 6 CSV files
ltm_full = load_ltm_instance(Path("../LTM/YQH26_data"))

print(f"Loaded LTM instance: {ltm_full.N} coverages, {ltm_full.M} packages, "
      f"{ltm_full.N * ltm_full.M} binary variables")
print(f"Families: {list(ltm_full.families.keys())}")
print(f"Mandatory families: {list(ltm_full.mandatory_families.keys())}")
print(f"Optional families: {list(ltm_full.optional_families.keys())}")
print(f"Price sensitivity beta: {ltm_full.price_sensitivity_beta}")

# Display all 20 coverages
df_cov = pd.DataFrame([
    {"name": c.name, "family": c.family, "price": c.price,
     "take_rate": c.take_rate, "margin_pct": c.contribution_margin_pct,
     "mandatory": c.is_mandatory_in_family}
    for c in ltm_full.coverages
])
df_cov

## 2. LTM Packages, Rules, and Segment Affinity

The LTM specification defines **10 customer-segment packages** with per-package discounts,
**6 dependency rules** (actuarially motivated), **3 incompatibility pairs**, and a
**segment affinity matrix** alpha_{i,m} that breaks package symmetry.

In [ ]:
# --- Packages ---
print("=== 10 Customer-Segment Packages ===")
df_pkg = pd.DataFrame([
    {"index": m, "name": ltm_full.package_names[m],
     "discount": ltm_full.package_discounts[m]}
    for m in range(ltm_full.M)
])
display(df_pkg)

# --- Dependency Rules ---
print(f"\n=== {len(ltm_full.dependency_rules)} Dependency Rules ===")
print("(dependent requires prerequisite in same package)")
for r in ltm_full.dependency_rules:
    print(f"  {r.dependent} --> requires {r.requires}")

# --- Incompatibility Pairs ---
print(f"\n=== {len(ltm_full.compatibility_rules)} Incompatibility Pairs ===")
print("(these two coverages cannot appear in the same package)")
for r in ltm_full.compatibility_rules:
    print(f"  {r.coverage_i} <-/-> {r.coverage_j}")

In [ ]:
# --- Segment Affinity Matrix alpha_{i,m} ---
# Values > 1.0 mean the segment has higher affinity for that coverage;
# values < 1.0 mean lower affinity. This breaks package symmetry in the ILP.
print("=== Segment Affinity Matrix (alpha_{i,m}) ===")
df_affinity = pd.DataFrame(
    ltm_full.segment_affinity,
    index=[c.name for c in ltm_full.coverages],
    columns=ltm_full.package_names,
)
# Style: highlight values far from 1.0
df_affinity.style.background_gradient(cmap="RdYlGn", vmin=0.7, vmax=1.3, axis=None)

## 3. Create Scaling Instances

The hackathon challenge defines a scaling hierarchy from toy to full size,
created by subsampling the full LTM instance:

| n_coverages | n_packages | n_vars | Purpose |
|---|---|---|---|
| 5 | 2 | 10 | Statevector simulation (DQI) |
| 7 | 3 | 21 | Small QAOA / DQI comparison |
| 10 | 2 | 20 | Mid-size benchmark |
| 10 | 5 | 50 | Larger circuit test |
| 20 | 10 | 200 | Full LTM reference instance |

In [ ]:
# Define scaling hierarchy: (n_coverages, n_packages, n_vars)
sizes = [(5, 2, 10), (7, 3, 21), (10, 2, 20), (10, 5, 50), (20, 10, 200)]

scaling_instances = {}
for n_cov, n_pkg, n_vars in sizes:
    if n_cov == ltm_full.N and n_pkg == ltm_full.M:
        prob = ltm_full  # Full instance, no subsampling needed
    else:
        prob = subsample_problem(ltm_full, n_coverages=n_cov, n_packages=n_pkg)

    scaling_instances[n_vars] = prob
    families = list(prob.families.keys())
    n_dep = len(prob.dependency_rules)
    n_incompat = len(prob.compatibility_rules)
    print(f"n={n_vars:>3d}  ({n_cov} cov x {n_pkg} pkg)  "
          f"families={len(families)}  deps={n_dep}  incompat={n_incompat}  "
          f"families: {families}")

## 4. Save Data & ILP Matrices

Export the LTM coverages to CSV and save ILP matrices (c, A, b) for each scaling instance
as `.npz` files for downstream use by the quantum algorithm notebooks.

In [ ]:
import os
os.makedirs("../data", exist_ok=True)

# Save all 20 LTM coverages to CSV
df_cov.to_csv("../data/ltm_coverages.csv", index=False)
print(f"Saved {len(df_cov)} LTM coverages to data/ltm_coverages.csv")

# Save ILP matrices for each scaling instance
for n_vars, prob in scaling_instances.items():
    c, A, b = get_ilp_matrices(prob)
    np.savez(
        f"../data/ilp_n{n_vars}.npz",
        c=c, A=A, b=b,
        num_coverages=prob.N,
        num_packages=prob.M,
    )
    print(f"\nn={n_vars}: c({c.shape}), A({A.shape}), b({b.shape})")
    print(f"  Cost vector range: [{c.min():.4f}, {c.max():.4f}]")
    print(f"  Constraint matrix (non-zero): {np.count_nonzero(A)} / {A.size}")

# Also save legacy filenames for backward compatibility
for old_name, n_vars in [("toy", 10), ("full", 200)]:
    data = np.load(f"../data/ilp_n{n_vars}.npz")
    np.savez(f"../data/ilp_{old_name}.npz", **dict(data))
    print(f"\nCopied ilp_n{n_vars}.npz -> ilp_{old_name}.npz")

## 5. Smoke Test: Solve Smallest Instance Classically

Run the classical ILP solver (PuLP/CBC) on the n=10 toy instance as a sanity check,
then summarize all scaling instance solutions.

In [ ]:
# Detailed solve of the n=10 toy instance
toy = scaling_instances[10]
result_toy = solve_ilp(toy)
print("=== n=10 Toy Instance (Classical Baseline) ===")
print(f"Status: {result_toy['status']}")
print(f"Objective: {result_toy['objective']:.4f}")
for m, pkg in enumerate(result_toy['packages']):
    pkg_name = toy.package_names[m] if toy.package_names else f"Package {m}"
    print(f"  {pkg_name}: {pkg}")
print(f"Solution vector: {result_toy['solution_vector']}")

# Solve all scaling instances and summarize
print("\n=== Classical Solutions Across All Scaling Instances ===")
rows = []
for n_vars in sorted(scaling_instances.keys()):
    prob = scaling_instances[n_vars]
    result = solve_ilp(prob)
    rows.append({
        "n_vars": n_vars,
        "n_cov": prob.N,
        "n_pkg": prob.M,
        "status": result["status"],
        "objective": round(result["objective"], 4),
        "total_selected": int(result["solution_vector"].sum()),
    })

df_results = pd.DataFrame(rows)
df_results